In [1]:
##### SVM LINEAR #####
# say we have the following example : two classes, A and B
# and the data is as follows after preprocessing : 
# AAAAAAAAAAAAAAAAAAAA
# A <- support vector
#
# ------------------------------------------------
#
# B <- support vector
# BBBBBBBBBBBBBBBBBBBB
# how do we split? we find the hyperplane which splits s.t. the minimum distance between the classes and the hyperplane is maximal
# this is what linear svm does (support vector machine)

import numpy as np
import pandas as pd
import kagglehub

x, y = None, None
dataset, model = None, None

def load_data():
    """
    Preprocessing function
    """
    global x, y
    global dataset, model

    dataset = pd.read_csv("/kaggle/input/datasets/organizations/uciml/breast-cancer-wisconsin-data/data.csv")
    y = np.where(dataset["diagnosis"] == "M", 1.0, -1.0)
    dataset = dataset.drop(columns=["diagnosis","id","Unnamed: 32"])
    x = np.array(dataset)
    
    print(y.shape)

class SVM():
    """
    Linear support vector machine class. First computes scores across all train data (linear func) then
    'checks' how well the score is (firstly it's random of course, weight and bias are trainable param).
    Finally, it looks for values that are negative or subunitary to train param.
    """
    def __init__(self, learning_rate=1e-2, lambbda=1e-2):
        self.w, self.b = None, None
        self.lr = learning_rate
        self.loss = None
        self.max_iter = 10**5
        self.lambd = lambbda # this is used as a factor in stabilisation

    def _hinge(self, ys):
        return np.maximum(0, 1 - ys) # for hinge loss
    
    def fit(self, x, y):
        self.b = 0
        m = x.shape[1]
        n = x.shape[0]
        self.w = np.zeros(m)

        for _ in range(self.max_iter):
            scores = x @ self.w + self.b # same formula as logistic regression
            margin = y * scores # we take the formula yi * scoresi to find how good of a guess it is. say we have y = -1 and score = -3. by multiplying, we notice that we get 3 which is supraunitary, therefore a good guess.
            mask = margin < 1 # we are only interested in negative and subunitary values. even if positive but under 1, it is too close to the hyperplane therefore we can do better.
            self.loss = 1/2 * self.lambd * np.dot(self.w, self.w) + np.mean(self._hinge(margin)) # hinge loss, np.dot = ||w||^2 where the norm is euclidean norm

            dw = self.lambd * self.w - x[mask].T @ y[mask] # dloss/dw, we use mask since we only care when margin < 1, and if we look at _hinge, we see its max of 0 or 1 - margin and since margin <1 we always choose the latter
            self.w = self.w - self.lr/n * dw

            db = -np.sum(y[mask])
            self.b = self.b - self.lr/n * db

    
    def predict(self, x):
        scores = x @ self.w + self.b # our classes are split into either 1 or -1. why not 0 and 1? well, unlike other algorithms, where the binary outcome is expressied via 0 or 1, we assume that our scores equation is equal to 0 (to find the best fit) and 1 = above, -1 = below.
        return np.where(scores >= 0, 1, -1)
        
load_data()

model = SVM()

def train_test_split(x, y, test_size=0.2):
    n = x.shape[0]

    indices = np.arange(n)
    np.random.shuffle(indices)

    split = int((1 - test_size) * n)

    train_idx = indices[:split]
    test_idx = indices[split:]

    return (
        x[train_idx],
        x[test_idx],
        y[train_idx],
        y[test_idx]
    )

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

x_train, x_test, y_train, y_test = train_test_split(x, y)
mean = np.mean(x_train, axis=0)
std = np.std(x_train, axis=0)

x_train = (x_train - mean) / std
x_test = (x_test - mean) / std
model.fit(x_train, y_train)

train_pred = model.predict(x_train)
test_pred = model.predict(x_test)

print(f"Train Accuracy: {accuracy(y_train, train_pred):.4f}")
print(f"Test Accuracy : {accuracy(y_test, test_pred):.4f}")

(569,)
Train Accuracy: 0.9912
Test Accuracy : 0.9737
